# Dataset Builder From Models

Andrew E. Davidson  
aedaivds@ucsc.edu 02/10/25  

Copyright (c) 2020-2023, Regents of the University of California All rights reserved.   https://polyformproject.org/licenses/noncommercial/1.0.0  

elifeBinaryRandomForestResults.ipynb trained set of binary random forest classifiers using the elife data and using best hyper parameters from our hyperparameter search.

This notebook TODO this is probably no longer true
1. loads each of the binary random forest models
2. gets the list of features used
3. loads the elife counts for these features
4. save the count data

This data can be use to evaluate a random forest multiclassifier or train a GAN

In [1]:
import ipynbname

# use display() to print an html version of a data frame
# useful if dataFrame output is not generated by last like of cell
from IPython.display import display

import joblib
import math
import numpy as np
import os
import pandas as pd
import pprint as pp
import sys

notebookName = ipynbname.name()
notebookPath = ipynbname.path()
notebookDir = os.path.dirname(notebookPath)

#outDir = f'{notebookDir}/{notebookName}.out'
outDir = f'/private/groups/kimlab/aedavids/elife/{notebookName}.out'
os.makedirs(outDir, exist_ok=True)
print(f'outDir:\n{outDir}')

dataOutDir = os.path.join(outDir, "data")
os.makedirs(dataOutDir, exist_ok=True)
print(f'\ndataOutDir ;\n{dataOutDir}')

import logging
#loglevel = "DEBUG"
#loglevel = "INFO"
loglevel = "WARN"
# logFMT = "%(asctime)s %(levelname)s [thr:%(threadName)s %(name)s %(funcName)s() line:%(lineno)s] [%(message)s]"
logFMT = "%(asctime)s %(levelname)s %(name)s %(funcName)s() line:%(lineno)s] [%(message)s]"
logging.basicConfig(format=logFMT, level=loglevel)    
logger = logging.getLogger(notebookName)

meaningOfLife = 42

outDir:
/private/groups/kimlab/aedavids/elife/datasetBuilderFromModels.out

dataOutDir ;
/private/groups/kimlab/aedavids/elife/datasetBuilderFromModels.out/data


In [2]:
# setting the python path allows us to run python scripts from using
# the CLI. 
ORIG_PYTHONPATH = os.environ['PYTHONPATH']

deconvolutionModules = notebookPath.parent.joinpath("../../../deconvolutionAnalysis/python/")
print("deconvolutionModules: {}\n".format(deconvolutionModules))

PYTHONPATH = ORIG_PYTHONPATH + f':{deconvolutionModules}'
print("PYTHONPATH: {}\n".format(PYTHONPATH))

intraExtraRNA_POCModules=notebookPath.parent.joinpath("../../python/src")
print("intraExtraRNA_POCModules: {}\n".format(intraExtraRNA_POCModules))

PYTHONPATH = PYTHONPATH + f':{intraExtraRNA_POCModules}'
print("PYTHONPATH: {}\n".format(PYTHONPATH))

########
os.environ["PYTHONPATH"] = PYTHONPATH
PYTHONPATH = os.environ["PYTHONPATH"]
print("PYTHONPATH: {}\n".format(PYTHONPATH))

# to be able to import our local python files we need to set the sys.path
# https://stackoverflow.com/a/50155834
sys.path.append( str(deconvolutionModules) )
sys.path.append( str(intraExtraRNA_POCModules) )
print("\nsys.path:\n{}\n".format(sys.path))

deconvolutionModules: /private/home/aedavids/extraCellularRNA/intraExtraRNA_POC/jupyterNotebooks/elife/../../../deconvolutionAnalysis/python

PYTHONPATH: :/private/home/aedavids/extraCellularRNA/src:/private/home/aedavids/extraCellularRNA/intraExtraRNA_POC/jupyterNotebooks/elife/../../../deconvolutionAnalysis/python

intraExtraRNA_POCModules: /private/home/aedavids/extraCellularRNA/intraExtraRNA_POC/jupyterNotebooks/elife/../../python/src

PYTHONPATH: :/private/home/aedavids/extraCellularRNA/src:/private/home/aedavids/extraCellularRNA/intraExtraRNA_POC/jupyterNotebooks/elife/../../../deconvolutionAnalysis/python:/private/home/aedavids/extraCellularRNA/intraExtraRNA_POC/jupyterNotebooks/elife/../../python/src

PYTHONPATH: :/private/home/aedavids/extraCellularRNA/src:/private/home/aedavids/extraCellularRNA/intraExtraRNA_POC/jupyterNotebooks/elife/../../../deconvolutionAnalysis/python:/private/home/aedavids/extraCellularRNA/intraExtraRNA_POC/jupyterNotebooks/elife/../../python/src


sys.p

In [3]:
notebookPath.parent

PosixPath('/private/home/aedavids/extraCellularRNA/intraExtraRNA_POC/jupyterNotebooks/elife')

In [4]:
# local imports
# from analysis.utilities import loadList
from intraExtraRNA.elifeUtilities import loadElifeTrainingData

In [5]:
%%time

pipelineStageName = "best10CuratedDegree1_ce467ff"

selectElifeCategories = [ "Colorectal Cancer", "Esophagus Cancer", "Healthy donor",
                         "Liver Cancer", "Lung Cancer", "Stomach Cancer"]

# Whole_Blood is considered a healthy control
# the binary classifiers extraCellularRNA/intraExtraRNA_POC/jupyterNotebooks/elife/elifeBinaryRandomForestResults.ipynb 
# use a single feature. for example "COAD" and did not include features from a healthy control
# the thought was 'COAD or not. Keep the number of features to a min'.
# the 'Lung" featues was a lump of see /private/groups/kimlab/GTEx_TCGA/1vsAllLumpBrain and 
# extraCellularRNA/deconvolutionAnalysis/lumpBrain
features = ["COAD", "ESCA",  "Esophagus_Mucosa", "Liver", "Lung", "Stomach", "Whole_Blood"]

t = loadElifeTrainingData(pipelineStageName,
                                 features,
                                 selectElifeCategories,
                                 )
HUGO_Genes, elifeGenes, missingGenes, countDF, metaDF, XDF, yNP, labelEncoder, mapDF = t

2025-02-18 15:39:42,205 INFO intraExtraRNA.elifeUtilities loadElifeTrainingData() line:232] [BEGIN]
2025-02-18 15:39:42,206 DEBUG intraExtraRNA.elifeUtilities loadElifeTrainingData() line:233] [AEDWIP pipelineStageName : best10CuratedDegree1_ce467ff]
2025-02-18 15:39:42,207 INFO intraExtraRNA.elifeUtilities loadCounts() line:156] [countPath : /private/groups/kimlab/alex/data/elife/elife_all_norm_counts_2023-05-18.csv]
2025-02-18 15:39:43,943 DEBUG intraExtraRNA.elifeUtilities loadElifeTrainingData() line:274] [AEDWIP category : COAD]
2025-02-18 15:39:43,946 INFO analysis.utilities findSignatureGenesForPipelineStage() line:199] [resultFile : /private/groups/kimlab/aedavids/deconvolution/1vsAll-~gender_category/best10CuratedDegree1_ce467ff/training/best10CuratedDegree1.sh.out/GTEx_TCGA-design-tilda_gender_category-padj-0001-lfc-20-n-10/COAD_vs_all.results]
2025-02-18 15:39:43,947 DEBUG analysis.utilities findSignatureGenesForPipelineStage() line:201] [numRowsToSkip : 0]
2025-02-18 15:39:

CPU times: user 1min 5s, sys: 9.15 s, total: 1min 14s
Wall time: 1min 14s


In [6]:
# print( missingGenes )
# mapDF
print(f'len(HUGO_Genes) : {len(HUGO_Genes)}')
# elifeGenes
# metaDF
print(f'XDF.shape : {XDF.shape}')
print( XDF.iloc[0:5, 0:6] )

len(HUGO_Genes) : 70
XDF.shape : (224, 70)
gene         ENSG00000117395.13  ENSG00000158714.11  ENSG00000180667.11  \
SRR14506659           74.463527            0.000000         1697.147893   
SRR14506660          229.237454           36.816924          566.147046   
SRR14506661            0.000000            0.000000          940.677707   
SRR14506662          520.926228            0.000000          356.854975   
SRR14506663          577.672082            0.000000         1018.732524   

gene         ENSG00000170385.10  ENSG00000134318.15  ENSG00000225889.10  
SRR14506659           65.155586          564.681749            0.000000  
SRR14506660          368.169245           40.984878            0.000000  
SRR14506661            0.000000          199.672155           48.808749  
SRR14506662            0.000000          319.938943            0.000000  
SRR14506663           81.966985           58.547846           25.370733  


In [7]:
aedwip

NameError: name 'aedwip' is not defined

# Get the list of features
load the binary random forest models and get their features

In [ ]:
ebrfrRootDir = f'/private/groups/kimlab/aedavids/elife/elifeBinaryRandomForestResults.out'
modelDir = os.path.join(ebrfrRootDir, "model")
print(f'modelDir :\n{modelDir}')

# model = joblib.load('random_forest_model.joblib')
